In [1]:
# !pip install mlflow scikit-learn pandas torch --quiet

import mlflow
import mlflow.sklearn
import pandas as pd
import torch
from torch import nn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

/home/vboxuser/Downloads/myvenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000


In [2]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target
y = y.astype(int)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def train_and_log_mlp(hidden_layer_sizes, learning_rate_init, max_iter, run_name):
    with mlflow.start_run(run_name = run_name):
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("learning_rate_init", learning_rate_init)

        model = MLPClassifier(
            hidden_layer_sizes = hidden_layer_sizes,
            learning_rate_init = learning_rate_init,
            max_iter = max_iter,
            random_state = 42
        )

        model.fit(X_train, y_train)
        train_acc = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_test, model.predict(X_test))
        train_loss = model.loss_

        mlflow.log_metric("Train accuracy", train_acc)
        mlflow.log_metric("Test accuracy", val_acc)
        mlflow.log_metric("Train loss", train_loss)

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  Train accuracy={train_acc:.4f}  Validation accuracy={val_acc:.4f}")
        return run_id


In [4]:
experiments = [
    ((32,), 0.1, 200, "exp1"),
    ((32,), 0.01, 400, "exp2"),
    ((64,), 0.1, 200, "exp3"),
    ((64,), 0.01, 400, "exp4"),
    ((64, 32), 0.1, 200, "exp5"),
    ((64, 32), 0.001, 400, "exp6")
]
sweep_run_ids = []
for hidden_layer, lr, max_iterations, run_name in experiments:
    rid = train_and_log_mlp(hidden_layer, lr, max_iterations, run_name)
    sweep_run_ids.append(rid)

Logged run 9130ffc42a1f4199ae1210cc059f0d5c  |  Train accuracy=0.0979  Validation accuracy=0.0997
🏃 View run exp1 at: http://localhost:5000/#/experiments/2/runs/9130ffc42a1f4199ae1210cc059f0d5c
🧪 View experiment at: http://localhost:5000/#/experiments/2
Logged run 16ff4f2c39d742dbbd11121559c0b9c4  |  Train accuracy=0.5772  Validation accuracy=0.5779
🏃 View run exp2 at: http://localhost:5000/#/experiments/2/runs/16ff4f2c39d742dbbd11121559c0b9c4
🧪 View experiment at: http://localhost:5000/#/experiments/2
Logged run 5de29adbbb9b42d69c7ffe62063a44f2  |  Train accuracy=0.1121  Validation accuracy=0.1143
🏃 View run exp3 at: http://localhost:5000/#/experiments/2/runs/5de29adbbb9b42d69c7ffe62063a44f2
🧪 View experiment at: http://localhost:5000/#/experiments/2
Logged run 34ed9dd087fb4fb38d94405dfc5965a3  |  Train accuracy=0.6997  Validation accuracy=0.6969
🏃 View run exp4 at: http://localhost:5000/#/experiments/2/runs/34ed9dd087fb4fb38d94405dfc5965a3
🧪 View experiment at: http://localhost:5000/